# Quad hose dataset (L, R, T, B) — folder: quad_hose_tracker

Loads sessions under `data_folder`:
```
session/
  images/
  hoseL.csv / hoseR.csv / hoseT.csv / hoseB.csv
```

**Training policy: all 4 or none**
- All four labels → four Gaussian heatmaps
- No labels → empty heatmaps (null / no hose)
- Any incomplete set (1–3 labels) → **discarded**

**Hose-relative labels** (reference: hose pointing down). When hose points up,
hoseL may have larger x than hoseR, and hoseT may have larger y than hoseB —
that is correct; labels are not swapped.


In [1]:
import random
import pickle
import sys
from pathlib import Path

import numpy as np

sys.path.append('../../..')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils/src')

from hose_data import load_hose_data, audit_hose_label_order


In [ ]:
data_folder = '/mnt/c/Users/wanglab/Desktop/pico-hose/'
output_folder = '/mnt/c/Users/wanglab/Desktop/pico-hose/'

target_resolution = (256, 256)
gaussian_sigma = (11, 11)  # tight kernels; L/R ~18px apart at 256

random_seed_value = 4
train_split = 0.8
max_null_ratio = 1.0  # at most 1 null per all-4 labeled frame

train_pkl = 'training_data_hose_lrtb.pkl'
test_pkl = 'testing_data_hose_lrtb.pkl'


In [3]:
print('Orientation mix (up-ish often = hose pointing up):')
for session, stats in audit_hose_label_order(data_folder).items():
    n = max(stats['labeled'], 1)
    print(
        f"  {session}: labeled={stats['labeled']} "
        f"Lx>Rx={stats['l_x_gt_r']} ({100*stats['l_x_gt_r']/n:.0f}%) "
        f"Ty>By={stats['t_y_gt_b']} ({100*stats['t_y_gt_b']/n:.0f}%)"
    )

training_images, training_image_filenames, training_labels = load_hose_data(
    data_folder,
    target_resolution=target_resolution,
    gaussian_sigma=gaussian_sigma,
    include_null_frames=True,
    require_all_four=True,  # discard frames with 1–3 labels
    max_null_ratio=max_null_ratio,
    null_seed=random_seed_value,
)

print(f"Images: {training_images.shape}")
print(f"Labels: {training_labels.shape}  # (N,H,W,4) = L,R,T,B")


Orientation mix (up-ish often = hose pointing up):
  102225_2: labeled=234 Lx>Rx=0 (0%) Ty>By=0 (0%)
  102525_1: labeled=176 Lx>Rx=80 (45%) Ty>By=80 (45%)


Loading sessions:   0%|          | 0/2 [00:00<?, ?it/s]

102225_2: 381 images, L=234, R=234, T=234, B=234, resolution=(480, 640)


Loading sessions:  50%|#####     | 1/2 [00:08<00:08,  8.87s/it]

102525_1: 215 images, L=176, R=176, T=176, B=176, resolution=(480, 640)


Loading sessions: 100%|##########| 2/2 [00:13<00:00,  6.86s/it]

Loaded 596 frames (all4=410, null=186, discarded_partial=0, unreadable=0)
Keypoint channels: ['hoseL', 'hoseR', 'hoseT', 'hoseB']
Images: (596, 256, 256, 3)
Labels: (596, 256, 256, 4)  # (N,H,W,4) = L,R,T,B


In [4]:
has_any = np.any(training_labels > 0, axis=(1, 2, 3))
n_ch = [
    int(np.any(training_labels[..., c] > 0, axis=(1, 2)).sum())
    for c in range(4)
]
print(f"Total frames: {len(training_image_filenames)}")
print(f"All-4 labeled: {int(has_any.sum())}")
print(f"Null: {int((~has_any).sum())}")
print(f"Per-channel nonzero counts (should match if all-or-none): L,R,T,B = {n_ch}")


Total frames: 596
All-4 labeled: 410
Null: 186
Per-channel nonzero counts (should match if all-or-none): L,R,T,B = [410, 410, 410, 410]


In [5]:
%matplotlib inline
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact

CHANNEL_NAMES = ['hoseL', 'hoseR', 'hoseT', 'hoseB']
COLORS = ['yellow', 'cyan', 'magenta', 'lime']

def show_example(idx=0):
    img = training_images[idx]
    labels = training_labels[idx]
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
    axes[0].imshow(img[..., ::-1])
    axes[0].set_title(Path(training_image_filenames[idx]).name)
    axes[0].axis('off')
    for c, (ax, name, color) in enumerate(zip(axes[1:], CHANNEL_NAMES, COLORS)):
        ax.imshow(img[..., ::-1])
        heat = labels[..., c]
        if np.any(heat > 0):
            cy, cx = np.unravel_index(int(np.argmax(heat)), heat.shape)
            ax.scatter(cx, cy, c=color, s=80, marker='x', linewidths=2)
            ax.set_title(f'{name} ({cx},{cy})')
        else:
            ax.set_title(f'{name}: none')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

slider = IntSlider(min=0, max=max(0, len(training_images)-1), step=1, value=0)
interact(show_example, idx=slider)


interactive(children=(IntSlider(value=0, description='idx', max=595), Output()), _dom_classes=('widget-interac…

<function __main__.show_example(idx=0)>

In [6]:
# Null frames for session 102225_2 (images with no L/R/T/B labels)
import re
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact
from IPython.display import display

null_session = '102225_2'
null_session_path = Path(data_folder) / null_session
null_img_dir = null_session_path / 'images'

labeled_frames = set()
for name in ['hoseL', 'hoseR', 'hoseT', 'hoseB']:
    csv_path = null_session_path / f'{name}.csv'
    labeled_frames |= set(pd.read_csv(csv_path)['frame'].astype(int))

null_entries = []  # (frame, filename, path)
for img_path in sorted(null_img_dir.glob('*.png')):
    m = re.search(r'img(\d+)', img_path.stem)
    if not m:
        continue
    frame = int(m.group(1))
    if frame not in labeled_frames:
        null_entries.append((frame, img_path.name, img_path))

print(f'{null_session}: {len(null_entries)} null frames (no keypoints)')
print('origin frames:')
for frame, fname, _ in null_entries:
    print(f'  frame {frame}  ->  {fname}')

display(pd.DataFrame(
    [{'origin_frame': f, 'image': n} for f, n, _ in null_entries]
))

def show_null(i=0):
    if not null_entries:
        print('No null frames')
        return
    frame, fname, path = null_entries[i]
    img = plt.imread(path)
    plt.figure(figsize=(5, 4))
    plt.imshow(img, cmap='gray' if img.ndim == 2 else None)
    plt.title(f'{null_session} null [{i+1}/{len(null_entries)}]\norigin frame {frame}  ({fname})')
    plt.axis('off')
    plt.show()

null_slider = IntSlider(
    min=0,
    max=max(0, len(null_entries) - 1),
    step=1,
    value=0,
    description='null idx',
)
interact(show_null, i=null_slider)


102225_2: 147 null frames (no keypoints)
origin frames:
  frame 7250  ->  img0007250.png
  frame 8000  ->  img0008000.png
  frame 8024  ->  img0008024.png
  frame 8039  ->  img0008039.png
  frame 8053  ->  img0008053.png
  frame 8068  ->  img0008068.png
  frame 8084  ->  img0008084.png
  frame 8098  ->  img0008098.png
  frame 19250  ->  img0019250.png
  frame 27500  ->  img0027500.png
  frame 28500  ->  img0028500.png
  frame 28518  ->  img0028518.png
  frame 28750  ->  img0028750.png
  frame 32750  ->  img0032750.png
  frame 34000  ->  img0034000.png
  frame 34001  ->  img0034001.png
  frame 34035  ->  img0034035.png
  frame 38500  ->  img0038500.png
  frame 40250  ->  img0040250.png
  frame 40277  ->  img0040277.png
  frame 40309  ->  img0040309.png
  frame 40500  ->  img0040500.png
  frame 41310  ->  img0041310.png
  frame 41500  ->  img0041500.png
  frame 42561  ->  img0042561.png
  frame 42590  ->  img0042590.png
  frame 42602  ->  img0042602.png
  frame 42615  ->  img0042615.png


,origin_frame,image
0,7250,img0007250.png
1,8000,img0008000.png
2,8024,img0008024.png
3,8039,img0008039.png
4,8053,img0008053.png
...,...,...
142,589387,img0589387.png
143,590750,img0590750.png
144,590773,img0590773.png
145,590783,img0590783.png


interactive(children=(IntSlider(value=0, description='null idx', max=146), Output()), _dom_classes=('widget-in…

<function __main__.show_null(i=0)>

In [7]:
shuffled_indexes = list(range(training_images.shape[0]))
random.Random(random_seed_value).shuffle(shuffled_indexes)
n_train = int(len(shuffled_indexes) * train_split)
training_indexes = shuffled_indexes[:n_train]
testing_indexes = shuffled_indexes[n_train:]

labels_for_save = np.asarray(training_labels)
assert labels_for_save.ndim == 4 and labels_for_save.shape[-1] == 4

out = Path(output_folder)
out.mkdir(parents=True, exist_ok=True)

with open(out / train_pkl, 'wb') as handle:
    pickle.dump(
        (training_images[training_indexes], labels_for_save[training_indexes]),
        handle,
    )
with open(out / test_pkl, 'wb') as handle:
    pickle.dump(
        (training_images[testing_indexes], labels_for_save[testing_indexes]),
        handle,
    )

print(f"Train: {len(training_indexes)}, Test: {len(testing_indexes)}")
print(f"Wrote {out / train_pkl}")
print(f"Wrote {out / test_pkl}")


Train: 476, Test: 120
Wrote /mnt/c/Users/wanglab/Desktop/pico-hose/training_data_hose.pkl
Wrote /mnt/c/Users/wanglab/Desktop/pico-hose/testing_data_hose.pkl
